# AIVORA Studio — Google Colab / ComfyUI 算力直连笔记本

利用 Google Colab（推荐 **L4 24GB** 或 **A100 40GB/80GB**）一键启动 **ComfyUI + AIVORA 安全鉴权桥接 + Cloudflare Quick Tunnel**，页面底部会自动输出一段「一键连接码」，复制粘贴进 **AIVORA Studio → 模型中心 → Google Colab / 本地 ComfyUI 算力直连** 即可自动填入全部配置。

### 显卡档位建议（请按实际分配显卡选择预设）
- **L4 (24GB VRAM)**：推荐 `flux-schnell`（生图）+ `wan2.2-i2v-5b` 或 `wan2.2-i2v-14b-gguf`（图生视频）
- **A100 (40GB / 80GB VRAM)**：全系列通吃，含 `ltx-2.5-distilled` 带原生音效视频
- **T4 (16GB VRAM)**：建议仅用于 `flux-schnell` FP8 生图

In [ ]:
# @title 1. 检查 GPU 显存并可选挂载 Google Drive（缓存模型权重，重开免重复下载）
USE_GOOGLE_DRIVE_CACHE = True  # @param {type:"boolean"}
MODEL_PRESET = "flux-schnell + wan2.2-i2v-5b"  # @param ["flux-schnell", "flux-schnell + wan2.2-i2v-5b", "flux-schnell + wan2.2-i2v-14b-gguf", "flux-schnell + ltx-2.5-distilled"]

import os
import subprocess

print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode())

MODELS_CACHE_DIR = "/content/aivora_models"
if USE_GOOGLE_DRIVE_CACHE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        MODELS_CACHE_DIR = "/content/drive/MyDrive/AIVORA_Models"
    except Exception as exc:
        print("非 Colab 环境或跳过 Drive 挂载：", exc)

os.makedirs(MODELS_CACHE_DIR, exist_ok=True)
print("模型缓存目录：", MODELS_CACHE_DIR)

In [ ]:
# @title 2. 安装 ComfyUI、依赖包与 AIVORA 桥接服务（首次约 3-6 分钟）
import os
import subprocess
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")
if not COMFY_DIR.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY_DIR)])

subprocess.check_call(["pip", "install", "-q", "-r", str(COMFY_DIR / "requirements.txt"), "fastapi", "uvicorn", "pydantic", "huggingface_hub"])

# 下载 cloudflared 用于生成临时 HTTPS 隧道
cf_bin = Path("/usr/local/bin/cloudflared")
if not cf_bin.exists():
    subprocess.check_call([
        "curl", "-fsSL",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-o", str(cf_bin),
    ])
    cf_bin.chmod(0o755)

# 拉取本仓库的 aivora_worker.py
worker_file = Path("/content/aivora_worker.py")
subprocess.check_call([
    "curl", "-fsSL",
    "https://raw.githubusercontent.com/chen11-A/aivora-studio/main/colab/aivora_worker.py",
    "-o", str(worker_file),
])

os.makedirs("/content/workflows", exist_ok=True)
print("安装完成！可将 ComfyUI 导出的 API Workflow JSON 放入 /content/workflows/<model_id>.json 以自定义节点图。")

In [ ]:
# @title 3. 一键启动 ComfyUI + AIVORA 鉴权 Worker + Cloudflare 临时隧道
import base64
import json
import os
import re
import secrets
import subprocess
import time
import urllib.request

AIVORA_TOKEN = secrets.token_urlsafe(24)
ADVERTISED = "flux-schnell,wan2.2-i2v-5b,wan2.2-i2v-14b-gguf,ltx-2.5-distilled"

# 1) 启动 ComfyUI 后端
comfy_proc = subprocess.Popen(
    ["python3", "/content/ComfyUI/main.py", "--listen", "127.0.0.1", "--port", "8188"],
    stdout=open("/content/comfy.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(45):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        break
    except Exception:
        time.sleep(2)

# 2) 启动 AIVORA 安全鉴权桥接（强制校验 32 位随机 Bearer Token）
env = os.environ.copy()
env["AIVORA_WORKER_TOKEN"] = AIVORA_TOKEN
env["AIVORA_WORKFLOW_DIR"] = "/content/workflows"
env["AIVORA_MODELS"] = ADVERTISED
worker_proc = subprocess.Popen(
    ["uvicorn", "aivora_worker:app", "--app-dir", "/content", "--host", "127.0.0.1", "--port", "8199"],
    env=env,
    stdout=open("/content/worker.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(3)

# 3) 启动 Cloudflare Quick Tunnel 并解析公网临时 HTTPS 网址
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8199", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
start_ts = time.time()
while time.time() - start_ts < 45 and tunnel_proc.stdout is not None:
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

connect_payload = {
    "url": public_url or "",
    "token": AIVORA_TOKEN,
    "models": [m.strip() for m in ADVERTISED.split(",") if m.strip()],
    "name": "Google Colab GPU 算力",
}
connect_code = "AIVORA-COMFY::" + base64.b64encode(
    json.dumps(connect_payload, ensure_ascii=False).encode("utf-8")
).decode("ascii")

print("=" * 72)
print("  AIVORA Studio 算力 Worker 已就绪！")
print("  【最简单用法】整段复制下方「一键连接码」，粘贴进模型中心即可自动填充：")
print("-" * 72)
print(connect_code)
print("-" * 72)
print(f"  Base URL     : {public_url or '请检查 /content/worker.log'}")
print(f"  Access Token : {AIVORA_TOKEN}")
print("=" * 72)
print("提示：保持本单元格运行即可；Colab 重开后点模型中心「编辑」再粘一次新连接码即可。")

try:
    worker_proc.wait()
except KeyboardInterrupt:
    tunnel_proc.terminate()
    worker_proc.terminate()
    comfy_proc.terminate()